In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile")
llm

c:\Users\anand\Desktop\GenAI - Krish Naik\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026331BA84A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002632D1B75F0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path="document.pdf")
documents = loader.load_and_split()
documents

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'document.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealth

In [3]:
from langchain_classic import PromptTemplate

template = """
  You are an expert text summarization assistant. Your task is to read the provided text carefully and produce a clear, accurate, and concise summary.
  Paragraph: {text}
"""

prompt = PromptTemplate(input_variables=["paragraph"], template=template)

In [4]:
from langchain_classic.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm=llm, chain_type="stuff", prompt=prompt, verbose=True)

response = chain.run(documents)
print(response)

C:\Users\anand\AppData\Local\Temp\ipykernel_3740\4106425260.py:5: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = chain.run(documents)




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

  You are an expert text summarization assistant. Your task is to read the provided text carefully and produce a clear, accurate, and concise summary.
  Paragraph: A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Ras

#### Map Reduce Text Summarization

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
documents

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'document.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealth

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
final_docs = splitter.split_documents(documents=documents)

In [8]:
final_docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'document.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealth

In [9]:
chunks_prompt = """ 
  Summarize the below speech:
  Speech: {text}
  Summary:
"""

map_prompt = PromptTemplate(input_variables=['text'], template=chunks_prompt)

final_prompt = """
  Provide the final summary of the entire speech with these important points.
  Add a motivational title and start the precise summary with an introduction and provide the summary in bullet points.
  Paragraph: {text}
"""

final_prompt_template = PromptTemplate(
  input_variables=["text"],
  template=final_prompt
)

summary_chain = load_summarize_chain(
  llm=llm,
  chain_type="map_reduce",
  map_prompt = map_prompt,
  combine_prompt=final_prompt_template
)

In [10]:
response = summary_chain.run(final_docs)
response

c:\Users\anand\Desktop\GenAI - Krish Naik\.venv\Lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


'**"Empowering a Brighter Future: The Vision for a Developed India"**\n\nAs the former President of India, A P J Abdul Kalam, concludes his five-year tenure at Rashtrapati Bhavan, he reflects on his experiences and shares a vision for a prosperous and proud nation. In his departing speech, he emphasizes the importance of empowering India\'s youth and villages to achieve a brighter future. The key points of his speech can be summarized as follows:\n\n* Empowering villages is crucial for India\'s development, and providing physical, electronic, and knowledge connectivity can transform villages and lead to transformation and competitiveness.\n* The aspirations of the young should guide the nation\'s development, and initiatives like PURA can generate employment and create entrepreneurs.\n* Accelerating development to meet the aspirations of the youth, empowering villages, and mobilizing rural core competence for competitiveness are essential for the country\'s progress.\n* Focusing on agr

In [11]:
refine_chain = load_summarize_chain(llm=llm, chain_type="refine")

response = refine_chain.run(final_docs)
response

'The refined summary remains largely the same as the original, with the new context providing a conclusion to A.P.J. Abdul Kalam\'s departing speech. Here is the refined summary:\n\nIn his departing speech, A.P.J. Abdul Kalam reflects on his 5-year tenure as President, expressing gratitude for the associations he made. He highlights 10 key messages, including accelerating development, empowering villages, and overcoming problems through partnership, with a focus on youth aspiration and a vision for a developed India by 2020. This vision is driven by the aspirations of the nation\'s 540 million youth, who dream of a prosperous, safe, and proud India. Kalam\'s experiences, such as his visit to Nagaland and the Periyar Maniammai College of Technology for Women\'s initiative to provide urban amenities in rural areas, have shaped his vision for empowering villages and mobilizing rural core competence for competitiveness, ultimately contributing to the development of India by 2020. The Periy